# The Calculus of Backpropagation

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 02.09 |
| Time | ~90 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/10_calculus_of_backpropagation.ipynb)

---

## 🎯 Learning Objective

Derive backpropagation from scratch for dense, convolutional, and attention layers, and learn to diagnose vanishing/exploding gradients from the resulting gradient-flow equations.

---

## 📐 Theory

This capstone notebook synthesizes every idea from `02.01`–`02.09` into the one algorithm that
trains virtually every modern neural network. There is no new mathematical machinery here —
only the chain rule (`02.04`), applied layer by layer, with careful bookkeeping.

### Backprop through a dense layer

A dense layer computes $\mathbf{z} = W\mathbf{x} + \mathbf{b}$, then an elementwise activation
$\mathbf{a} = \sigma(\mathbf{z})$. Given the upstream gradient $\partial L/\partial\mathbf{a}$,
the chain rule (in Jacobian form, from `02.06`) gives:

$$\frac{\partial L}{\partial \mathbf{z}} = \frac{\partial L}{\partial \mathbf{a}} \odot \sigma'(\mathbf{z}), \quad
\frac{\partial L}{\partial W} = \frac{\partial L}{\partial \mathbf{z}}\,\mathbf{x}^\top, \quad
\frac{\partial L}{\partial \mathbf{b}} = \frac{\partial L}{\partial \mathbf{z}}, \quad
\frac{\partial L}{\partial \mathbf{x}} = W^\top \frac{\partial L}{\partial \mathbf{z}}$$

($\odot$ is elementwise product.) Each is the local-gradient recipe from `02.04`: the
activation's local gradient is $\sigma'(\mathbf{z})$; the linear map's local gradient w.r.t.
each of $W,\mathbf{b},\mathbf{x}$ follows from the multivariate chain rule. Crucially,
$\partial L/\partial\mathbf{x}$ is exactly the upstream gradient the *previous* layer
receives — this is what makes stacking layers work: each layer only backprops through itself.

### Backprop through convolution and attention

A convolution is a dense layer with shared, sparse weights; its backward pass convolves the
upstream gradient with the *flipped* kernel to get the input gradient, and convolves the input
with the upstream gradient to get the kernel gradient — same local-gradient logic, sparser map.

Attention computes $\mathbf{o} = \text{softmax}(QK^\top/\sqrt{d_k})V$ (recall `01.10`).
Backprop here has three receiving ends ($Q$, $K$, $V$), but the mechanism is identical:
propagate the upstream gradient back through the product with $V$, then through softmax
(softmax is NOT elementwise, so its local gradient is a full matrix per row, not a vector),
then through $QK^\top$. Nothing new — the chain rule applied to more matrix products in sequence.

### Gradient flow: vanishing and exploding gradients

Chaining $n$ layers multiplies $n$ local Jacobians together (`02.04`'s reverse-mode recipe).
If each layer's local gradient has singular values consistently below $1$ (sigmoid's
derivative maxes out at $0.25$), the product shrinks geometrically with depth — **vanishing
gradients**: early layers get a signal too small to learn from. If local gradients are
consistently above $1$ (large weights, unsaturated ReLU), the product grows geometrically —
**exploding gradients**. Both are mechanical consequences of multiplying many numbers
repeatedly, nothing more. This explains why architectures use residual connections (an
identity path with local gradient exactly $1$, regardless of depth), why LayerNorm/BatchNorm
exist (control activation scale so local gradients stay well-behaved), and why non-saturating
activations like ReLU replaced sigmoid/tanh in deep networks.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Tracking gradient norm layer-by-layer as it backpropagates through a deep stack makes vanishing
and exploding gradients visible directly: a sigmoid stack's gradient norm collapses toward the
input layer, while an unnormalized ReLU stack's gradient norm can blow up instead.

In [ ]:
# Track ||gradient|| layer-by-layer as it backpropagates through a deep stack of random
# layers, for sigmoid (saturating) vs ReLU (non-saturating) activations.
rng = np.random.default_rng(0)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def gradient_norms_through_depth(n_layers, activation, width=20, weight_scale=0.5):
    Ws = [rng.standard_normal((width, width)) * weight_scale for _ in range(n_layers)]
    x = rng.standard_normal(width)
    activs, zs = [x], []
    for W in Ws:
        z = activs[-1] @ W
        zs.append(z)
        activs.append(sigmoid(z) if activation == "sigmoid" else np.maximum(0, z))

    grad = np.ones(width)          # dL/d(final activation), for a toy loss L = sum(activation)
    norms = [np.linalg.norm(grad)]
    for i in reversed(range(n_layers)):
        local_grad = activs[i + 1] * (1 - activs[i + 1]) if activation == "sigmoid" else (zs[i] > 0)
        grad = (grad * local_grad) @ Ws[i].T   # exactly the dense-layer backward formula above
        norms.append(np.linalg.norm(grad))
    return norms[::-1]   # index 0 = gradient that reaches the INPUT layer

n_layers = 20
sigmoid_norms = gradient_norms_through_depth(n_layers, "sigmoid")
relu_norms = gradient_norms_through_depth(n_layers, "relu")

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(sigmoid_norms, 'o-', color="#C44E52", label="sigmoid (vanishing)")
ax.semilogy(relu_norms, 'o-', color="#4C72B0", label="ReLU, unnormalized weights (exploding)")
ax.set_xlabel("layer depth (0 = input side, 20 = output/loss side)")
ax.set_ylabel("||gradient|| (log scale)")
ax.set_title("Gradient magnitude across depth: the SAME chain-rule product,\ntwo very different outcomes")
ax.legend()
plt.show()

print(f"Sigmoid: gradient at output layer = {sigmoid_norms[-1]:.4f}, at input layer = {sigmoid_norms[0]:.2e}")
print(f"ReLU:    gradient at output layer = {relu_norms[-1]:.4f}, at input layer = {relu_norms[0]:.2e}")
print("\nBoth start from the SAME upstream gradient at the loss -- depth alone, compounded by")
print("the chain rule, is enough to shrink one to numerical zero and grow the other by 5 orders of magnitude.")

## 🐍 Implementation from Scratch

We implement forward AND backward passes for a dense layer and for self-attention -- by hand,
using only the formulas derived in the Theory section -- and verify every single gradient
against PyTorch's autograd. If our from-scratch derivation is correct, every number must match
exactly.

In [ ]:
import torch

# ---------- Dense layer: forward + backward, straight from the Theory section's formulas ----------
def dense_forward(x, W, b):
    z = x @ W + b
    return np.maximum(0, z), z  # ReLU

def dense_backward(dL_da, x, W, z):
    dL_dz = dL_da * (z > 0)     # chain through ReLU's local gradient
    dL_dW = np.outer(x, dL_dz)  # dL/dW = x . dL/dz^T
    dL_db = dL_dz
    dL_dx = dL_dz @ W.T          # becomes the UPSTREAM gradient for the previous layer
    return dL_dx, dL_dW, dL_db

rng = np.random.default_rng(0)
x0 = rng.standard_normal(4)
W0 = rng.standard_normal((4, 3)) * 0.5
b0 = rng.standard_normal(3) * 0.1
a0, z0 = dense_forward(x0, W0, b0)
dL_dx0, dL_dW0, dL_db0 = dense_backward(2 * a0, x0, W0, z0)  # dL/da for L = sum(a^2)

xt, Wt, bt = (torch.tensor(v, requires_grad=True) for v in (x0, W0, b0))
torch.sum(torch.relu(xt @ Wt + bt) ** 2).backward()
print("Dense layer backward, hand-derived vs. torch.autograd:")
print(f"  dL/dW match: {np.allclose(dL_dW0, Wt.grad.numpy())}")
print(f"  dL/db match: {np.allclose(dL_db0, bt.grad.numpy())}")
print(f"  dL/dx match: {np.allclose(dL_dx0, xt.grad.numpy())}")

# ---------- Self-attention: forward + backward for Q, K, V ----------
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)  # log-sum-exp-style shift, previewed in 06.02
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention_forward(Q, K, V):
    scale = 1 / np.sqrt(Q.shape[-1])
    A = softmax(Q @ K.T * scale, axis=-1)
    return A @ V, A, scale

def attention_backward(dL_dO, A, Q, K, V, scale):
    dL_dA = dL_dO @ V.T
    dL_dV = A.T @ dL_dO
    # softmax's local gradient is a full per-row Jacobian, not elementwise -- this closed
    # form comes from differentiating softmax(scores)_i w.r.t. every score_j
    dL_dscores = A * (dL_dA - np.sum(A * dL_dA, axis=-1, keepdims=True))
    return (dL_dscores @ K) * scale, (dL_dscores.T @ Q) * scale, dL_dV

n, d_k = 4, 8
Q0, K0, V0 = (rng.standard_normal((n, d_k)) * 0.3 for _ in range(3))
O0, A0, scale0 = attention_forward(Q0, K0, V0)
dL_dQ0, dL_dK0, dL_dV0 = attention_backward(2 * O0, A0, Q0, K0, V0, scale0)

Qt, Kt, Vt = (torch.tensor(m, requires_grad=True) for m in (Q0, K0, V0))
At = torch.softmax(Qt @ Kt.T * scale0, dim=-1)
torch.sum((At @ Vt) ** 2).backward()
print("\nSelf-attention backward, hand-derived vs. torch.autograd:")
print(f"  dL/dQ match: {np.allclose(dL_dQ0, Qt.grad.numpy(), atol=1e-6)}")
print(f"  dL/dK match: {np.allclose(dL_dK0, Kt.grad.numpy(), atol=1e-6)}")
print(f"  dL/dV match: {np.allclose(dL_dV0, Vt.grad.numpy(), atol=1e-6)}")

## 🤖 Why This Matters for AI

Every equation above — dense backward, attention backward, gradient flow through depth — is
literally what happens inside `loss.backward()` for a real transformer, just scaled up. The
vanishing/exploding analysis explains a design choice in every modern architecture: the
**residual connection**, $\mathbf{y}=\mathbf{x}+f(\mathbf{x})$. Backpropagating through it
splits the upstream gradient down two paths — through $f$, and along the identity, whose local
gradient is exactly $1$ regardless of $f$. Below, we rebuild the vanishing-gradient experiment
with residual connections added to the sigmoid stack, and confirm the input-layer gradient no
longer collapses — proof, via the chain rule alone, of why residuals make deep nets trainable.

In [ ]:
# Does a residual connection actually rescue the vanishing gradient from the Visual Intuition
# cell? Rebuild the SAME sigmoid stack, but each layer now computes y = x + sigmoid(x @ W).
def gradient_norms_residual(n_layers, width=20, weight_scale=0.5):
    Ws = [rng.standard_normal((width, width)) * weight_scale for _ in range(n_layers)]
    x = rng.standard_normal(width)
    activs, zs = [x], []
    for W in Ws:
        z = activs[-1] @ W
        zs.append(z)
        activs.append(activs[-1] + sigmoid(z))   # RESIDUAL: y = x + sigmoid(x @ W)

    grad = np.ones(width)
    norms = [np.linalg.norm(grad)]
    for i in reversed(range(n_layers)):
        s = sigmoid(zs[i])
        local_grad_f = s * (1 - s)
        # dL/dx = dL/dy . (I + local_grad_f-weighted W^T) -- the "+1" from the identity
        # path is what keeps this from collapsing toward zero the way the plain stack did
        grad = grad + (grad * local_grad_f) @ Ws[i].T
        norms.append(np.linalg.norm(grad))
    return norms[::-1]

residual_norms = gradient_norms_residual(n_layers)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(sigmoid_norms, 'o-', color="#C44E52", label="plain sigmoid stack (vanishes)")
ax.semilogy(residual_norms, 'o-', color="#55A868", label="sigmoid stack + residual connections")
ax.set_xlabel("layer depth (0 = input side, 20 = output/loss side)")
ax.set_ylabel("||gradient|| (log scale)")
ax.set_title("Residual connections: same depth, same activation,\nno more vanishing gradient")
ax.legend()
plt.show()

print(f"Plain sigmoid stack:    gradient at input layer = {sigmoid_norms[0]:.2e}")
print(f"Residual sigmoid stack: gradient at input layer = {residual_norms[0]:.4f}")
print(f"\nThe identity path's local gradient is exactly 1 -- adding it back at every layer prevents")
print(f"the repeated multiplication by small numbers (sigmoid'(z) <= 0.25) from ever fully collapsing the signal.")

## 🏋️ Exercises

### Warm-up
1. For a dense layer with $x=(1,-1)$, $W=\begin{pmatrix}0.5&-0.2\\0.3&0.8\end{pmatrix}$,
   $b=(0,0)$, ReLU, and $\partial L/\partial\mathbf{a}=(1,1)$, compute $\partial L/\partial W$
   by hand using this notebook's formulas. Verify with `dense_backward`.

### Practice
2. Extend `dense_forward`/`dense_backward` to a 2-layer network (dense -> ReLU -> dense) and
   verify the FULL gradient against `torch.autograd` on a random input.

### Challenge
3. Sweep `weight_scale` in `gradient_norms_through_depth` from 0.1 to 2.0 (`n_layers=20`) for
   the ReLU stack, and plot input-layer gradient norm vs. `weight_scale`. Find, approximately,
   the scale where the gradient neither vanishes nor explodes, and explain how this previews
   the motivation behind He/Glorot initialization.

---

## 📚 Further Reading
- Rumelhart, Hinton & Williams, ["Learning representations by back-propagating errors"](https://www.nature.com/articles/323533a0) (1986)
- He et al., ["Deep Residual Learning for Image Recognition"](https://arxiv.org/abs/1512.03385) (ResNet)
- Vaswani et al., ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762)
- Karpathy, [micrograd](https://github.com/karpathy/micrograd) (revisit alongside `02.04`'s `Value` engine)

---

*Next notebook: [Probability Fundamentals](../03_Probability_and_Statistics/01_probability_fundamentals.ipynb)*